# Backend API Tester — MTS AI Hub
Тестирует все эндпоинты. Запусти бэкенд: `uvicorn app.main:app --port 8000`

In [ ]:
import httpx, json
from pathlib import Path

BASE = "http://localhost:8000"
USER_ID = "test_user"

def ok(label, resp):
    icon = '✅' if resp.status_code < 400 else '❌'
    print(f"{icon} [{resp.status_code}] {label}")
    try: print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:600])
    except: print(resp.text[:400])
    print()

## 1. Health Check

In [ ]:
r = httpx.get(f"{BASE}/v1/health")
ok("Health Check", r)

## 2. Chat — Text (mws-gpt-alpha)

In [ ]:
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False, "user": USER_ID
}, timeout=30)
ok("Chat / Text (mws-gpt-alpha)", r)

## 3. Chat — Code (qwen3-coder-480b-a35b)

In [ ]:
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False, "user": USER_ID
}, timeout=30)
ok("Chat / Code (qwen3-coder-480b-a35b)", r)

## 4. Chat — Long Context (qwen2.5-72b-instruct)

In [ ]:
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen2.5-72b-instruct",
    "messages": [{"role": "user", "content": "Объясни принципы работы трансформерных архитектур подробно"}],
    "stream": False, "user": USER_ID
}, timeout=60)
ok("Chat / Long Context (qwen2.5-72b-instruct)", r)

## 5. Chat — Auto Router (model=auto)

In [ ]:
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Напиши алгоритм бинарного поиска"}],
    "stream": False, "user": USER_ID
}, timeout=30)
ok("Chat / Auto Router → ожидается qwen3-coder", r)

## 6. Chat — Streaming (SSE)

In [ ]:
print("🔄 Streaming:")
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи короткий анекдот"}],
    "stream": True, "user": USER_ID
}, timeout=60) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                delta = json.loads(line[6:])["choices"][0]["delta"].get("content", "")
                if delta: chunks.append(delta); print(delta, end="", flush=True)
            except: pass
print(f"\n\n✅ chunks: {len(chunks)}")

## 7. Embeddings (bge-m3)

In [ ]:
r = httpx.post(f"{BASE}/v1/embeddings", json={"model": "bge-m3", "input": "Тестовый текст"}, timeout=15)
ok("Embeddings (bge-m3)", r)

## 8. List Models

In [ ]:
r = httpx.get(f"{BASE}/v1/models")
ok("List Models", r)

## 9. Deep Research (SSE)

In [ ]:
print("🔄 Deep Research:")
with httpx.stream("POST", f"{BASE}/v1/research", json={"query": "Квантовые вычисления", "user_id": USER_ID}, timeout=120) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line: print(line[:120])
print("\n✅ Research done")

## 10. Web Search

In [ ]:
r = httpx.post(f"{BASE}/v1/tools/web-search", json={"query": "FastAPI Python 2025", "max_results": 3}, timeout=20)
ok("Web Search", r)

## 11. Web Parse

In [ ]:
r = httpx.post(f"{BASE}/v1/tools/web-parse", json={"url": "https://example.com"}, timeout=20)
ok("Web Parse", r)

## 12. File Upload (TXT)

In [ ]:
test_txt = Path("/tmp/test_doc.txt")
test_txt.write_text("Это тестовый документ. Python — мощный язык. FastAPI очень быстрый.", encoding="utf-8")
with open(test_txt, "rb") as f:
    r = httpx.post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": USER_ID}, timeout=30)
ok("File Upload (TXT)", r)

## 13. File QA (chat с файлом)

In [ ]:
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Что написано в моём документе?"}],
    "stream": False, "user": USER_ID,
    "attachments": [{"name": "test_doc.txt", "mime": "text/plain"}]
}, timeout=30)
ok("Chat / File QA", r)

## 14. Memory — GET

In [ ]:
r = httpx.get(f"{BASE}/v1/memory/{USER_ID}", timeout=10)
ok("Memory GET", r)

## 15. Memory — Search

In [ ]:
r = httpx.post(f"{BASE}/v1/memory/{USER_ID}/search", json={"query": "Python", "top_k": 5}, timeout=10)
ok("Memory Search", r)

## 16. Image Generation

In [ ]:
r = httpx.post(f"{BASE}/v1/image/generate", json={"prompt": "закат над морем", "width": 512, "height": 512}, timeout=60)
ok("Image Generate (fallback если SD недоступен)", r)

## 17. VLM Analyze

In [ ]:
r = httpx.post(f"{BASE}/v1/vlm/analyze", json={
    "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
    "question": "Что изображено?"
}, timeout=30)
ok("VLM Analyze (fallback если VLM недоступен)", r)

## 18. PPTX Generation

In [ ]:
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json={
    "topic": "Искусственный интеллект в 2025 году",
    "slide_count": 5, "language": "ru"
}, timeout=60)
if r.status_code == 200:
    out = Path("/tmp/test_output.pptx")
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved → {out} ({len(r.content):,} bytes)")
else:
    ok("PPTX Generate", r)

## 19. Voice Message (ASR)

In [ ]:
r = httpx.post(f"{BASE}/v1/voice/message",
    files={"audio": ("test.wav", b"RIFF", "audio/wav")},
    data={"user_id": USER_ID}, timeout=30)
icon = '✅ endpoint reachable' if r.status_code < 500 else '❌ server error'
print(f"{icon} [{r.status_code}] Voice Message")
print(r.text[:300])

## 20. Router Detection — все типы задач

In [ ]:
cases = [
    ("Привет, как дела?",                         "text",          "mws-gpt-alpha"),
    ("Напиши функцию сортировки на Python",        "code",          "qwen3-coder-480b-a35b"),
    ("Исследуй тему квантовых вычислений",         "deep_research", "qwen2.5-72b-instruct"),
    ("Зайди на https://example.com и расскажи",    "web_parse",     "mws-gpt-alpha"),
    ("Нарисуй закат над морем",                    "image_gen",     "qwen-image"),
    ("Глубокий анализ квантовых компьютеров",      "deep_research", "qwen2.5-72b-instruct"),
    ("Реализуй алгоритм Дейкстры на Go",           "code",          "qwen3-coder-480b-a35b"),
]
print(f"{'Ожидаемый тип':<18} {'Ожидаемая модель':<30} Статус | Сообщение")
print("-" * 85)
for msg, exp_task, exp_model in cases:
    try:
        r = httpx.post(f"{BASE}/v1/chat/completions", json={
            "model": "auto",
            "messages": [{"role": "user", "content": msg}],
            "stream": False, "user": USER_ID
        }, timeout=15)
        icon = '✅' if r.status_code < 400 else '❌'
        print(f"{exp_task:<18} {exp_model:<30} {icon} [{r.status_code}] | {msg[:45]}")
    except Exception as e:
        print(f"{exp_task:<18} {exp_model:<30} ❌ ERROR: {e}")

## 21. Summary — All Endpoints

In [ ]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/embeddings",                {"model":"bge-m3","input":"test"}),
    ("GET",  "/v1/models",                    None),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slide_count":3,"language":"ru"}),
    ("GET",  f"/v1/memory/{USER_ID}",         None),
    ("POST", f"/v1/memory/{USER_ID}/search",  {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat"}),
    ("POST", "/v1/vlm/analyze",               {"image_url":"https://example.com/img.jpg","question":"что?"}),
]
print(f"{'Method':<6} {'Endpoint':<40} {'Status':>6}  Result")
print("-" * 65)
passed = 0
for method, path, body in endpoints:
    try:
        r = httpx.get(f"{BASE}{path}", timeout=15) if method == "GET" else httpx.post(f"{BASE}{path}", json=body, timeout=20)
        icon = "✅" if r.status_code < 400 else "⚠️ "
        if r.status_code < 400: passed += 1
        print(f"{method:<6} {path:<40} {r.status_code:>6}  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<40} {'ERR':>6}  ❌ {str(e)[:35]}")
print(f"\n{'='*65}\nИтого: {passed}/{len(endpoints)} ОК")